# 00 · Framework, Business Problem & Data Set-up
**Fraud & AML analytics with statistics — from data gathering to deployment**

| Role | Book | How we use it |
|---|---|---|
| **Framework / steps** | *Fraud Analytics Using Descriptive, Predictive and Social Network Techniques* (Baesens, Van Vlasselaer, Verbeke) | The **analytics process model** (Fig 1.6): identify problem → identify & select data → clean → transform → analyse (descriptive / predictive / social network) → interpret, evaluate, deploy |
| **Concepts** | *AML Transaction Monitoring Systems Implementation* (Chau, van Dijck Nemcsik) | Scenarios, thresholds, alert productivity, segmentation, above/below-the-line testing, what-if & mock investigations |
| **Technical guide** | *Practical Statistics for Data Scientists* (Bruce, Bruce, Gedeck) | EDA, sampling & bias, bootstrap, hypothesis tests, regression, classification, ML, unsupervised learning |

## The roadmap (one notebook per step of the process model)

| # | Notebook | Process-model step | AML concept | Statistics used |
|---|---|---|---|---|
| 00 | this one | Identify business problem | Risk-based approach, cost of errors | Expected cost, base rates |
| 01 | `01_data_gathering_quality_sampling` | Identify sources · Select · Clean | Data quality, reconciliation | Random / stratified sampling, selection bias, bootstrap, sample size & power |
| 02 | `02_eda_cleaning_transformation` | Explore · Transform | Red flags, expected activity | Robust location/spread, QQ-plots, chi-square, Benford, outliers, WoE/IV, PCA |
| 03 | `03_descriptive_analytics` | Analyse (descriptive) | Peer groups, segmentation | Clustering, break-point tests, isolation forest, association lift |
| 03a | `03a_ch2_data_collection_sampling_preprocessing` | Prepare (concept deep-dive) | **Fraud Analytics Ch. 2** – all 18 concepts | Sampling, Benford, missing values, outliers, red flags, WoE/IV, filters, PCA, RIDIT/PRIDIT, segmentation |
| 03b | `03b_ch3_descriptive_analytics_concepts` | Analyse (concept deep-dive) | **Fraud Analytics Ch. 3** – all concepts | OLAP, Grubbs/Mahalanobis, break-point, peer group, association rules, hierarchical/k-means/SOM, constraints, one-class SVM |
| 03c | `03c_ch4_predictive_analytics_concepts` | Analyse (concept deep-dive) | **Fraud Analytics Ch. 4** – all concepts | Scorecards, trees, NN, SVM, ensembles, multiclass, evaluation, SMOTE, Pluto–Tasche, cost-sensitive learning |
| 03d | `03d_ch5_social_network_analytics_concepts` | Analyse (concept deep-dive) | **Fraud Analytics Ch. 5** – all concepts | Neighbourhood/centrality metrics, PageRank, ICA/Gibbs/LBP, modularity, bipartite graphs, Gotcha!-style pipeline |
| 04 | `04_aml_threshold_tuning_workflow` | Analyse · Evaluate | **Ch. 9 – 11 steps of threshold tuning** | Percentiles, Wilson CIs, chi-square, what-if, binary search |
| 05 | `05_threshold_tuning_statistical_critique` | Evaluate | Circularity, false negatives, overfitting | Bias, base-rate fallacy, regression to the mean, target shuffling, multiple testing |
| 06 | `06_predictive_analytics` | Analyse (predictive) | Alert scoring, false-positive reduction | Logistic reg., trees/boosting, ROC/PR/lift, imbalance, calibration |
| 07 | `07_social_network_analytics` | Analyse (social network) | Mule rings, shell counterparties | Graph metrics, community mining, permutation tests |
| 08 | `08_evaluation_deployment_monitoring` | Interpret · Evaluate · **Deploy** | Model risk, documentation | PSI/SSI, bootstrap traffic lights, binomial calibration test, champion-challenger |

> **Iterate!** The process model is a loop, not a line: findings in 03–07 will send you back to 01–02 (new data, better features). Write that down in your project log.

In [1]:
import sys, platform
import numpy as np, pandas as pd, scipy, sklearn, matplotlib, seaborn as sns, networkx as nx, statsmodels
import matplotlib.pyplot as plt
import amlkit as ak

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")

print("python", platform.python_version())
for m in (np, pd, scipy, sklearn, matplotlib, nx, statsmodels):
    print(f"{m.__name__:12s}{m.__version__}")

python 3.12.3
numpy       2.4.4
pandas      3.0.2
scipy       1.17.1
sklearn     1.8.0
matplotlib  3.10.8
networkx    3.6.1
statsmodels 0.15.0


## Step 1 · Identify the business problem
The book stresses that **the business problem, not the algorithm, drives every later choice** (which data, which metric, what "good" means). Write it down before touching data.

We work two use cases side by side, because Fraud and AML differ in *what the label is* and *who decides*:

| | **A. Payment (card-not-present) fraud** | **B. AML transaction monitoring** |
|---|---|---|
| Decision | Block / step-up authenticate a transaction **now** | Open an alert for an analyst; maybe file an STR later |
| Unit of analysis | Transaction | Customer / alert |
| Label | Chargeback / confirmed fraud (relatively fast & reliable) | STR filed (slow, **biased**: only alerted cases are ever investigated) |
| Time to decide | Milliseconds | Days–weeks |
| Cost of a miss (FN) | Loss amount (measurable) | Regulatory / reputational risk (hard to measure) |
| Cost of a false alarm (FP) | Customer friction | Analyst hours |

In [2]:
problem = {
    "A_card_fraud": dict(
        question="Which card-not-present transactions should be declined or challenged?",
        unit="transaction", label="is_fraud (chargeback-confirmed)",
        cost_fp=3.0,      # USD friction + support cost of wrongly challenging a good txn
        cost_fn=None,     # = the transaction amount (loss) -> handled per transaction
        success="Maximise fraud value caught at <= 1% challenge rate",
    ),
    "B_aml_alerts": dict(
        question="Which customers/alerts deserve analyst time, and are our scenario thresholds defensible?",
        unit="customer / alert", label="STR filed after investigation (biased!) ; oracle exists only in this lab",
        cost_fp=45.0,     # USD analyst time to close an unproductive alert (assumption - replace with yours)
        cost_fn=5000.0,   # USD proxy for regulatory + reputational exposure of one missed case (assumption)
        success="Higher alert productivity WITHOUT losing coverage of true suspicious activity (false negatives)",
    ),
}
pd.DataFrame(problem).T

,question,unit,label,cost_fp,cost_fn,success
A_card_fraud,Which card-not-present transactions should be ...,transaction,is_fraud (chargeback-confirmed),3.00,None,Maximise fraud value caught at <= 1% challenge...
B_aml_alerts,"Which customers/alerts deserve analyst time, a...",customer / alert,STR filed after investigation (biased!) ; orac...,45.00,"5,000.00",Higher alert productivity WITHOUT losing cover...


In [3]:
def expected_cost(tp, fp, fn, tn, c_fp, c_fn, c_tp=0.0):
    '''Total cost of a decision rule. The AML book stresses cost when discussing true/false positive/negative (Ch. 9).'''
    return fp * c_fp + fn * c_fn + tp * c_tp

# toy illustration: same model, two operating points, 100,000 customers, 1% truly suspicious
N, prev = 100_000, 0.01
P, Nn = int(N * prev), int(N * (1 - prev))
for label, tpr, fpr in [("strict threshold", 0.60, 0.010), ("loose threshold", 0.90, 0.060)]:
    tp, fn = int(P * tpr), int(P * (1 - tpr))
    fp, tn = int(Nn * fpr), int(Nn * (1 - fpr))
    ppv = tp / (tp + fp)
    cost = expected_cost(tp, fp, fn, tn, 45, 5000)
    print(f"{label:17s} alerts={tp+fp:6,d}  precision={ppv:5.1%}  missed={fn:4d}  total cost=${cost:,.0f}")

strict threshold  alerts= 1,590  precision=37.7%  missed= 400  total cost=$2,044,550
loose threshold   alerts= 6,840  precision=13.2%  missed=  99  total cost=$762,300


**Read-across for later notebooks.** With 1 % prevalence, even a *good* rule has low precision (base-rate effect – *Practical Statistics* ch. 5 "rare class problem"). A loose threshold can cost **less overall** once you price the misses. Threshold tuning is therefore a *cost-and-risk* decision, not a search for the highest precision.

## Step 2 · Identify data sources — generate the lab data
We use a **synthetic bank** (3,500 customers, ≈0.8 M transactions, one year) so we can *know the truth* and therefore measure how good each statistical method really is. Real data has no such oracle — that is exactly why the AML book insists on below-the-line testing and mock investigations.

The generator deliberately injects real-world mess:
* duplicate transaction rows (ETL replays), KHR-denominated amounts, zero/negative amounts
* missing KYC values, duplicate KYC records, inconsistent segment labels
* five laundering typologies: **structuring**, **smurfing_low** (low-and-slow, *below* the rule's single-transaction floor), **pass_through**, **mule_ring**, **commingling** (structural break in cash deposits)
* card-not-present fraud bursts (night-time, foreign, new device)

In [4]:
ak.ensure_data()                      # writes data/raw/*.csv* the first time only
raw, kyc = ak.load_raw()
print("transactions:", raw.shape, "| KYC:", kyc.shape)
raw.head()

transactions: (823101, 10) | KYC: (3515, 8)


,txn_id,customer_id,ts,channel,amount,currency,counterparty_id,country,new_device,is_fraud
0,5,3491,2025-01-01,card_pos,175.36,USD,-1,KH,0,0
1,1,1159,2025-01-01,cash_deposit,"2,448.17",USD,-1,KH,0,0
2,4,2572,2025-01-01,transfer_out,"1,410.50",USD,1809,KH,0,0
3,2,1238,2025-01-01,cash_withdrawal,"1,143,900.00",KHR,-1,KH,0,0
4,3,2486,2025-01-01,cash_withdrawal,725.77,USD,-1,KH,0,0


In [5]:
data_dictionary = pd.DataFrame([
    ("transactions", "txn_id", "Transaction key (should be unique - it is not!)"),
    ("transactions", "customer_id", "FK to KYC"),
    ("transactions", "ts", "Transaction timestamp"),
    ("transactions", "channel", "cash_deposit, cash_withdrawal, wire_in/out, transfer_in/out, card_pos, card_online"),
    ("transactions", "amount / currency", "Amount in ORIGINAL currency (USD or KHR) - must be normalised"),
    ("transactions", "counterparty_id", "Other party (customer id if < 3500, external id otherwise, -1 = none)"),
    ("transactions", "country", "Counterparty / merchant country; HR_* = placeholder high-risk jurisdictions"),
    ("transactions", "new_device", "Card-online only: first time this device is seen"),
    ("transactions", "is_fraud", "Card fraud label (chargeback-confirmed)"),
    ("customers_kyc", "segment", "retail_salaried, retail_self_employed, small_business, cash_intensive, corporate"),
    ("customers_kyc", "declared_monthly_turnover", "Expected activity declared at onboarding (USD)"),
    ("customers_kyc", "tenure_years / region / country_risk / is_pep / community_id", "KYC attributes"),
    ("_hidden_ground_truth", "is_launderer, typology, ring_id", "ORACLE - used only to grade methods, never as a model input"),
], columns=["table", "column", "meaning"])
data_dictionary

,table,column,meaning
0,transactions,txn_id,Transaction key (should be unique - it is not!)
1,transactions,customer_id,FK to KYC
2,transactions,ts,Transaction timestamp
3,transactions,channel,"cash_deposit, cash_withdrawal, wire_in/out, tr..."
4,transactions,amount / currency,Amount in ORIGINAL currency (USD or KHR) - mus...
5,transactions,counterparty_id,"Other party (customer id if < 3500, external i..."
6,transactions,country,Counterparty / merchant country; HR_* = placeh...
7,transactions,new_device,Card-online only: first time this device is seen
8,transactions,is_fraud,Card fraud label (chargeback-confirmed)
9,customers_kyc,segment,"retail_salaried, retail_self_employed, small_b..."


In [6]:
truth = ak.load_truth()
prev_l = truth.is_launderer.mean()
fraud_rate = raw.loc[raw.channel == "card_online", "is_fraud"].mean()
print(f"Customers who are truly suspicious : {truth.is_launderer.sum():4d} ({prev_l:.2%} of customers)")
print(f"Card-online txns that are fraud    : {raw.is_fraud.sum():4d} ({fraud_rate:.3%} of card-online txns)")
truth[truth.is_launderer == 1].typology.value_counts().to_frame("customers")

Customers who are truly suspicious :  123 (3.51% of customers)
Card-online txns that are fraud    :  383 (0.452% of card-online txns)


,customers
typology,
structuring,47
mule_ring,23
smurfing_low,20
pass_through,19
commingling,14


> **Prevalence caveat.** Real portfolios have far fewer suspicious customers (often well below 1 %). The lab uses 3.5 % so that every experiment has enough positives to be statistically readable; notebook 05 shows what happens to precision at realistic prevalence.

### Your log for this step
Fill this in with **your own bank's** answers before you replace the synthetic extracts:

1. Which decision will the model/threshold change, and who acts on it?
2. What is the label, how is it produced, and **who never gets labelled** (selection bias)?
3. What are the costs of FP and FN? Who owns the risk appetite?
4. What is the minimum acceptable performance, and how will it be monitored after go-live?
5. Which data sources are needed and do we have lawful access? (privacy / RACI – Fraud Analytics ch. 7)

**Next → `01_data_gathering_quality_sampling`**